# Method 1 — Bandit Static Analysis (real tool, not a trained proxy)

**Fixes Reviewer #3, M2:** the previous version of this notebook trained a
Random Forest on 25 hand-engineered features and reported *that* as
"Bandit" -- Bandit itself was never actually run for the numbers in the
paper. That made Bandit an invalid baseline (a trained classifier standing
in for a rule engine), and made the DQN's evaluation questionable too,
since its most predictive feature was engineered to mimic Bandit's own
rules against a Bandit-flavored label.

**This version actually runs the Bandit CLI** on every sample. There is no
training step -- Bandit is a static rule engine, not a learned model. The
only thing tuned is a single decision threshold on the validation split,
which is standard practice for turning any continuous score into a binary
decision and does not make Bandit "learned."

**Outputs saved:**
- `results/bandit_metrics.json` -- accuracy, precision, recall, F1, AUC, confusion matrix
- `results/bandit_confusion_matrix.png` -- confusion matrix plot
- `models/bandit_scores.csv` -- per-sample Bandit score for every split (`id`, `split`, `bandit_score`), so `Hybrid.ipynb` can reuse these without re-running Bandit (one subprocess call per sample is slow at scale)
- `models/bandit_threshold.json` -- the single tuned threshold value


In [1]:
!pip install pandas numpy matplotlib seaborn bandit tqdm scikit-learn --quiet



[notice] A new release of pip available: 22.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ── Imports ───────────────────────────────────────────────────────────────
import os, sys, json, subprocess, tempfile, warnings, pickle as pkl
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report)

warnings.filterwarnings('ignore')
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f'Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M")}')


Timestamp: 2026-09-10 12:06


In [3]:
# ── Load dataset ──────────────────────────────────────────────────────────
CSV = 'dataset_with_content.csv'
if not os.path.exists(CSV):
    raise FileNotFoundError('Run the dataset-collection notebook first to generate dataset_with_content.csv')

df = pd.read_csv(CSV)
df.columns = df.columns.str.strip().str.lower()
df = df.dropna(subset=['content', 'label']).reset_index(drop=True)
df['label'] = df['label'].astype(int)
if 'id' not in df.columns:
    df['id'] = df.index

print(f'Dataset loaded: {len(df):,} samples')
print(f'Vulnerable: {(df["label"]==1).sum():,}   Safe: {(df["label"]==0).sum():,}')


Dataset loaded: 14,010 samples
Vulnerable: 2,357   Safe: 11,653


In [4]:
# ── Train / Val / Test split (same seed as every other notebook) ─────────
idx_all = np.arange(len(df))
y_all = df['label'].values

idx_tv, idx_test = train_test_split(idx_all, test_size=0.20, random_state=42, stratify=y_all)
idx_train, idx_val = train_test_split(idx_tv, test_size=0.25, random_state=42, stratify=y_all[idx_tv])

print(f'Train: {len(idx_train):,}   Val: {len(idx_val):,}   Test: {len(idx_test):,}')


Train: 8,406   Val: 2,802   Test: 2,802


## Run the real Bandit CLI

Each sample's code is written to a temp `.py` file, scanned with
`python -m bandit -f json -ll <file>`, and converted to a severity-weighted
risk score in [0, 1]. This is the *only* thing that produces a "Bandit
score" anywhere in this project -- no classifier stands in for it.


In [5]:
# ── Real Bandit CLI scoring ────────────────────────────────────────────────
SEV_WEIGHT = {'HIGH': 1.0, 'MEDIUM': 0.5, 'LOW': 0.2}

def bandit_score(code_str: str) -> float:
    """Runs the actual Bandit tool on one code sample and returns a
    severity-weighted risk score in [0, 1]. No training happens anywhere in
    this function -- Bandit is a static rule engine, this just runs it."""
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False,
                                          encoding='utf-8') as tmp:
            tmp.write(code_str)
            tmp_path = tmp.name
        result = subprocess.run(
            [sys.executable, '-m', 'bandit', '-f', 'json', '-ll', tmp_path],
            capture_output=True, text=True, timeout=15)
        os.unlink(tmp_path)
        data = json.loads(result.stdout)
        issues = data.get('results', [])
        total = sum(SEV_WEIGHT.get(i.get('issue_severity', 'LOW').upper(), 0.2)
                    for i in issues)
        return min(1.0, total / (len(issues) * 0.5 + 2)) if issues else 0.0
    except Exception:
        return 0.0

def score_split(indices, tag):
    scores = np.zeros(len(indices))
    for k, i in enumerate(tqdm(indices, desc=f'Bandit scan ({tag})')):
        scores[k] = bandit_score(df['content'].iloc[i])
    return scores

bandit_val_scores = score_split(idx_val, 'val')
bandit_test_scores = score_split(idx_test, 'test')
y_val = y_all[idx_val]
y_test = y_all[idx_test]

print(f'\nMean Bandit score -- val: {bandit_val_scores.mean():.3f}, '
      f'test: {bandit_test_scores.mean():.3f}')


Bandit scan (test): 100%|██████████| 2802/2802 [57:48<00:00,  1.24s/it]  


Mean Bandit score -- val: 0.021, test: 0.020


## Tune the decision threshold (validation only)

Bandit outputs a continuous risk score; turning that into a Vulnerable/Safe
decision needs a cutoff. The threshold is selected on the **validation**
split only, maximizing F1 -- the test split is never touched until the
final evaluation below.


In [6]:
# ── Threshold tuning on validation set ────────────────────────────────────
best_thr, best_f1 = 0.5, -1.0
for thr in np.arange(0.02, 0.98, 0.01):
    preds = (bandit_val_scores >= thr).astype(int)
    f1v = f1_score(y_val, preds, zero_division=0)
    if f1v > best_f1:
        best_f1, best_thr = f1v, thr

print(f'Tuned threshold: {best_thr:.2f}  (validation F1 = {best_f1:.4f})')

with open('models/bandit_threshold.json', 'w') as f:
    json.dump({'threshold': round(float(best_thr), 4),
               'validation_f1_at_threshold': round(float(best_f1), 4)}, f, indent=2)


Tuned threshold: 0.02  (validation F1 = 0.3124)


In [7]:
# ── Test set evaluation ────────────────────────────────────────────────────
test_preds = (bandit_test_scores >= best_thr).astype(int)

acc = accuracy_score(y_test, test_preds)
prec = precision_score(y_test, test_preds, zero_division=0)
rec = recall_score(y_test, test_preds, zero_division=0)
f1 = f1_score(y_test, test_preds, zero_division=0)
try:
    auc = roc_auc_score(y_test, bandit_test_scores)
except Exception:
    auc = float('nan')

print('=== Bandit (real CLI, rule-based, no training) — Test Set ===')
print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'AUC-ROC   : {auc:.4f}')
print()
print(classification_report(y_test, test_preds, target_names=['Safe', 'Vulnerable']))


=== Bandit (real CLI, rule-based, no training) — Test Set ===
Accuracy  : 0.8380
Precision : 0.5486
Recall    : 0.2038
F1 Score  : 0.2972
AUC-ROC   : 0.5854

              precision    recall  f1-score   support

        Safe       0.86      0.97      0.91      2331
  Vulnerable       0.55      0.20      0.30       471

    accuracy                           0.84      2802
   macro avg       0.70      0.58      0.60      2802
weighted avg       0.81      0.84      0.81      2802



In [8]:
# ── Confusion Matrix ────────────────────────────────────────────────────────
BG  = '#0f172a'
AXB = '#1e293b'
CT  = '#f1f5f9'
CG  = '#334155'

cm = confusion_matrix(y_test, test_preds)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5), facecolor=BG)
ax.set_facecolor(AXB)
sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='RdYlGn',
            xticklabels=['Safe', 'Vulnerable'], yticklabels=['Safe', 'Vulnerable'],
            cbar=False, annot_kws={'size': 14, 'color': CT})
ax.set_xlabel('Predicted', color=CT)
ax.set_ylabel('Actual', color=CT)
ax.set_title('Bandit (real CLI) — Confusion Matrix', color=CT, fontweight='bold')
ax.tick_params(colors=CT)
plt.tight_layout()
plt.savefig('results/bandit_confusion_matrix.png', dpi=300, facecolor=BG, bbox_inches='tight')
plt.show()
print('Saved: results/bandit_confusion_matrix.png')


Saved: results/bandit_confusion_matrix.png


In [ ]:
# ── ROC Curve ────────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, roc_thresholds = roc_curve(y_test, bandit_test_scores)
roc_auc_value = roc_auc_score(y_test, bandit_test_scores)

fig, ax = plt.subplots(figsize=(6, 6), facecolor=BG)
ax.set_facecolor(AXB)
ax.plot(fpr, tpr, color='#3b82f6', linewidth=2.5,
        label=f'Bandit (AUC = {roc_auc_value:.4f})')
ax.plot([0, 1], [0, 1], linestyle='--', color=CG, label='Random guess (AUC = 0.50)')
ax.set_xlabel('False Positive Rate', color=CT)
ax.set_ylabel('True Positive Rate', color=CT)
ax.set_title('Bandit (real CLI) — ROC Curve', color=CT, fontweight='bold')
ax.tick_params(colors=CT)
ax.legend(loc='lower right', facecolor=AXB, labelcolor=CT)
ax.grid(alpha=0.2, color=CG)
plt.tight_layout()
plt.savefig('results/bandit_roc_curve.png', dpi=300, facecolor=BG, bbox_inches='tight')
plt.show()
print('Saved: results/bandit_roc_curve.png')

## Save scores for every split (for `Hybrid.ipynb`)

Bandit is slow to re-run (one subprocess per sample), so the ensemble
notebook should load these precomputed scores rather than re-scanning.
Includes the **train** split too, in case a downstream step wants it, even
though Bandit itself never "trains" on it.


In [ ]:
# ── Score the train split too (for completeness / downstream reuse) ──────
bandit_train_scores = score_split(idx_train, 'train')

scores_df = pd.concat([
    pd.DataFrame({'id': df['id'].iloc[idx_train].values, 'split': 'train',
                  'bandit_score': bandit_train_scores}),
    pd.DataFrame({'id': df['id'].iloc[idx_val].values, 'split': 'val',
                  'bandit_score': bandit_val_scores}),
    pd.DataFrame({'id': df['id'].iloc[idx_test].values, 'split': 'test',
                  'bandit_score': bandit_test_scores}),
], ignore_index=True)

scores_df.to_csv('models/bandit_scores.csv', index=False)
print(f'Saved: models/bandit_scores.csv  ({len(scores_df):,} rows)')


Bandit scan (train):  79%|███████▉  | 6638/8406 [2:23:37<3:04:43,  6.27s/it]

In [ ]:
# ── Save metrics ────────────────────────────────────────────────────────────
metrics = {
    'method': 'Bandit (real CLI, rule-based, no training)',
    'threshold_tuned_on_val': round(float(best_thr), 4),
    'accuracy': round(float(acc), 4),
    'precision': round(float(prec), 4),
    'recall': round(float(rec), 4),
    'f1': round(float(f1), 4),
    'auc': round(float(auc), 4) if not np.isnan(auc) else None,
    'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
    'test_size': int(len(y_test)),
    'timestamp': datetime.now().isoformat(),
}
with open('results/bandit_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved: results/bandit_metrics.json')
print(json.dumps(metrics, indent=2))


## Next steps

1. `LLMs.ipynb` (CodeBERT) and `RL.ipynb` (Q-Learning/DQN) can run
   unchanged -- they already read from `dataset_with_content.csv`
   independently of this notebook.
2. `Hybrid.ipynb` should load `models/bandit_scores.csv` (join on `id`) as
   the Bandit component of the ensemble, instead of loading a pickled RF
   model -- there is no `bandit_rf_model.pkl` in this version, by design.
3. Table IV / Section VI in the paper should cite these numbers as the
   Bandit-only baseline; the method column should say "Bandit (rule-based
   static analysis)", not "Bandit / Random Forest."
